# 05. Multi-Turn Chat & Python Function Calling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/05_chat_and_function_calling.ipynb)

The `google-genai` SDK turns stateless model requests into stateful, tool-augmented assistants using two built-in primitives: **`client.chats.create()`** for managing multi-turn conversation history, and **Automatic Python Function Calling (`tools=[fn]`)**, where the SDK introspects Python type hints and docstrings, executes local functions when Gemini requests them, and feeds the structured results back automatically.

![Figure 5-1: Automatic Python Function Calling Execution Loop in google-genai](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_function_calling_loop.png)

*Figure 5-1: Automatic Python Function Calling Execution Loop in google-genai*

---

## 1. Client Setup & Stateful Multi-Turn Chat (`client.chats.create`)

Define `MODEL = "gemini-3.8-flash"` at the top of the cell, then read your `GEMINI_API_KEY` from Google Colab Secrets (`userdata.get`) to initialize a unified `genai.Client`:

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

While `client.models.generate_content()` is stateless, **`client.chats.create()`** maintains an in-memory `list[types.Content]` transcript across turns. Each call to `chat.send_message()` appends a `Content(role="user")` message, sends the accumulated history to Gemini, and records the returned `Content(role="model")` turn (along with any intermediate tool calls). Because multi-turn sessions resend a growing shared prefix on every turn, they automatically benefit from Gemini 3.8's **Implicit Context Caching**, lowering time-to-first-token (TTFT) and input token cost on repeated prefixes with zero extra configuration.

![Figure 5-2: Multi-Turn Chat History Accumulation and Tool-Augmented Session Flow](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_chat_state_context.png)

*Figure 5-2: Multi-Turn Chat History Accumulation and Tool-Augmented Session Flow*

Create a stateful chat session with `MODEL` (`gemini-3.8-flash`), send two short messages, and inspect `chat.get_history()` to verify how `user` and `model` turns accumulate:

In [ ]:
chat = client.chats.create(model=MODEL)

reply_1 = chat.send_message("My favorite programming language is Python.")
print("Turn 1:", reply_1.text)

reply_2 = chat.send_message("What is my favorite language, in one sentence?")
print("Turn 2:", reply_2.text)

print("\n--- Accumulated Session History ---")
for idx, turn in enumerate(chat.get_history(), start=1):
    preview = " ".join(
        p.text.strip() for p in (turn.parts or []) if p.text and not p.thought
    )
    print(f"[{idx}] role={turn.role:<5} | {preview[:80]}")

---

## 2. Automatic Python Function Calling & Tool-Enabled Chat

To give Gemini live access to external metrics, databases, or internal APIs, pass plain typed Python functions directly into **`types.GenerateContentConfig(tools=[...])`**. Include **type annotations** (`region: str -> dict[str, float]`) and a clear **Google-style docstring** (`Args:` and `Returns:`): the SDK compiles them into a JSON Schema tool declaration automatically.

First, call `client.models.generate_content(model=MODEL, ...)` with a one-line prompt comparing two regions. `genai.Client` executes `get_region_latency()` locally for each requested region, feeds the returned dictionaries back as `FunctionResponse` payloads, and prints the grounded comparison:

In [ ]:
from google.genai import types

REGION_LATENCY_MS: dict[str, dict[str, float]] = {
    "asia-northeast1": {"p50_ms": 14.2, "p95_ms": 31.5},
    "us-central1": {"p50_ms": 24.5, "p95_ms": 58.4},
    "europe-west1": {"p50_ms": 38.6, "p95_ms": 84.2},
}


def get_region_latency(region: str) -> dict[str, float]:
    """Fetch live p50 and p95 API latency in milliseconds for a cloud region.

    Args:
        region: Cloud region ID ('asia-northeast1', 'us-central1', or 'europe-west1').

    Returns:
        Dictionary with 'p50_ms' median and 'p95_ms' tail latency values.
    """
    return REGION_LATENCY_MS.get(region, {"p50_ms": -1.0, "p95_ms": -1.0})


response = client.models.generate_content(
    model=MODEL,
    contents="Compare latency in asia-northeast1 and us-central1.",
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)
print(response.text)

Next, combine both primitives by passing `tools=[get_region_latency]` into **`client.chats.create(model=MODEL, config=...)`**. Across follow-up turns, the session remembers earlier regions while invoking new tool calls on demand and recording every `FunctionCall` and `FunctionResponse` inside `tool_chat.get_history()`:

In [ ]:
tool_chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)

turn_1 = tool_chat.send_message("Check latency in asia-northeast1 and us-central1.")
print("Turn 1:\n", turn_1.text)

turn_2 = tool_chat.send_message("Check europe-west1 too. Which is fastest, in one sentence?")
print("\nTurn 2:\n", turn_2.text)

print("\n--- Recorded Tool Call Trace ---")
for turn in tool_chat.get_history():
    for part in turn.parts or []:
        if part.function_call:
            fc = part.function_call
            print(f"-> FunctionCall: {fc.name}({dict(fc.args or {})})")
        elif part.function_response:
            print(f"<- FunctionResponse: {part.function_response.response}")

> **Key Takeaways**: Define `MODEL = "gemini-3.8-flash"` once at the top of your notebook, use `client.chats.create(model=MODEL)` for stateful multi-turn context, pass typed Python functions into `tools=[fn]` for automatic function calling with `generate_content()`, and combine both in a tool-enabled chat session to handle multi-step follow-up questions with zero manual tool-loop boilerplate.